<h1> Train Test Split

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

df_u_clean2 = pd.read_csv(
    '/content/drive/MyDrive/df_u_clean2.csv'
)

print(df_u_clean2.head())

In [ ]:
from sklearn.model_selection import train_test_split

X = df_u_clean2['narrative']
y = df_u_clean2['Timely response?']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
#validation and final test sets from test set
from sklearn.model_selection import train_test_split

X_val, X_test, y_val, y_test = train_test_split(
    X_test,
    y_test,
    test_size=0.5,
    stratify=y_test,
    random_state=42
)

<h1> 1. random undersampling

In [ ]:
import pandas as pd

# Combine X_train and y_train into one dataframe temporarily
train_df = pd.concat(
    [X_train.reset_index(drop=True),
     y_train.reset_index(drop=True)],
    axis=1
)

# Separate the classes
yes_df = train_df[train_df['Timely response?'] == 'Yes']
no_df = train_df[train_df['Timely response?'] == 'No']

print("Before undersampling:")
print(train_df['Timely response?'].value_counts())

# Random undersampling
yes_under = yes_df.sample(
    n=len(no_df),
    random_state=42
)

# Combine and shuffle
train_under = pd.concat([yes_under, no_df])

train_under = train_under.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("\nAfter undersampling:")
print(train_under['Timely response?'].value_counts())

# Separate X and y again
X_train_under = train_under['narrative']
y_train_under = train_under['Timely response?']

<H1>Encoding using sentence-transformers/all-MiniLM-L6-v2 model on X_train X_test

In [ ]:
! pip install scikit-learn

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
#from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# 1. Initialize the embedding model
# This will automatically download the model on the first run and cache it
print("Loading MiniLM model...")
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# 2. Convert text to embeddings
# NOTE: sentence-transformers expects a list of strings.
# If X_train_under is a Pandas Series, convert it using .tolist()
print("Generating embeddings for the training set (this might take a moment)...")
X_train_text = X_train_under.tolist() if hasattr(X_train_under, 'tolist') else list(X_train_under)
X_train = embedding_model.encode(X_train_text, batch_size=400, show_progress_bar=True)

print("Generating embeddings for the test set...")
X_test_text = (
    X_test.tolist()
    if hasattr(X_test, 'tolist')
    else list(X_test)
)

X_test = embedding_model.encode(
    X_test_text,
    batch_size=200,
    show_progress_bar=True
)

print("Generating embeddings for the validation set...")
X_val_text = (
    X_val.tolist()
    if hasattr(X_val, 'tolist')
    else list(X_val)
)

X_val = embedding_model.encode(
    X_val_text,
    batch_size=128,
    show_progress_bar=True
)

# Quick sanity check on shapes
print(f"\nTraining embeddings shape: {X_train.shape}")
print(f"Validation embeddings shape: {X_val.shape}")
print(f"Test embeddings shape: {X_test.shape}")

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Initialize the encoder
le = LabelEncoder()

# Fit ONLY on training labels
y_train = le.fit_transform(y_train_under)

# Transform validation and test labels
y_val = le.transform(y_val)

y_test = le.transform(y_test)   # or y_test if you didn't rename it

<h1> Model Training & Testing

In [ ]:
!pip install dagshub mlflow

In [ ]:
import dagshub
import mlflow

mlflow.set_tracking_uri('')
dagshub.init(repo_owner='', repo_name='', mlflow=True)

# mlflow.set_experiment("Logistic Regression Baseline")
mlflow.set_experiment("undersample_hyperparam_model_urgent-")


In [ ]:
!pip install optuna

In [ ]:
import optuna
import mlflow
import mlflow.sklearn
import time

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


<h2>Step 1: Create the objective function

In [ ]:
def objective_logistic(trial):

    with mlflow.start_run(
        run_name=f"Trial_{trial.number}",
        nested=True
    ):

        start_time = time.time()

        # =====================================
        # Tag
        # =====================================

        mlflow.set_tag(
            "model",
            "LogisticRegression"
        )

        # =====================================
        # Hyperparameters
        # =====================================

        C = trial.suggest_float(
            "C",
            1e-3,
            100,
            log=True
        )

        class_weight = trial.suggest_categorical(
            "class_weight",
            [
                None,
                "balanced"
            ]
        )

        # =====================================
        # Model
        # =====================================

        model = LogisticRegression(

            C=C,

            solver='lbfgs',

            penalty='l2',

            class_weight=class_weight,

            max_iter=1000,

            random_state=42

        )

        # =====================================
        # Train
        # =====================================

        model.fit(

            X_train,

            y_train

        )

        # =====================================
        # Predict on Validation Set
        # =====================================

        y_pred = model.predict(

            X_val

        )

        # =====================================
        # Metrics
        # =====================================

        accuracy = accuracy_score(

            y_val,

            y_pred

        )

        precision = precision_score(

            y_val,

            y_pred

        )

        recall = recall_score(

            y_val,

            y_pred

        )

        f1 = f1_score(

            y_val,

            y_pred

        )

        training_time = time.time() - start_time

        # =====================================
        # MLflow Logging
        # =====================================

        mlflow.log_params({

            "C": C,

            "class_weight": class_weight,

            "solver": "lbfgs",

            "penalty": "l2",

            "max_iter": 1000

        })

        mlflow.log_metrics({

            "accuracy": accuracy,

            "precision": precision,

            "recall": recall,

            "f1_score": f1,

            "training_time_seconds": training_time

        })

        return f1

<h2>Step 2: Create a reusable tuning function

In [ ]:
def tune_logistic(n_trials=50):

    with mlflow.start_run(

        run_name="LogisticRegression_Optuna"

    ):

        mlflow.log_param(

            "embedding_model",

            "all-MiniLM-L6-v2"

        )

        mlflow.log_param(

            "embedding_dimension",

            384

        )

        mlflow.log_param(

            "n_trials",

            n_trials

        )

        study = optuna.create_study(

            direction="maximize",

            study_name="LogisticRegression"

        )

        study.optimize(

            objective_logistic,

            n_trials=n_trials

        )

        # ==============================
        # Best Trial
        # ==============================

        best_params = study.best_params

        best_f1 = study.best_value

        print("\nBest Trial")

        print("----------------------")

        print(

            "Best Validation F1:",

            best_f1

        )

        print("\nBest Parameters:")

        for k, v in best_params.items():

            print(f"{k} : {v}")

        # ==============================
        # Train Best Model
        # ==============================

        best_model = LogisticRegression(

            **best_params,

            solver='lbfgs',

            penalty='l2',

            max_iter=1000,

            random_state=42

        )

        best_model.fit(

            X_train,

            y_train

        )

        # =====================================
        # Evaluate Best Model on Validation Set
        # =====================================

        y_pred_best = best_model.predict(

            X_val

        )

        best_accuracy = accuracy_score(

            y_val,

            y_pred_best

        )

        best_precision = precision_score(

            y_val,

            y_pred_best

        )

        best_recall = recall_score(

            y_val,

            y_pred_best

        )

        best_val_f1 = f1_score(

            y_val,

            y_pred_best

        )

        # =====================================
        # Log Best Parameters
        # =====================================

        mlflow.log_params({

            f"best_{k}": v

            for k, v in best_params.items()

        })

        # =====================================
        # Log Best Metrics
        # =====================================

        mlflow.log_metrics({

            "best_validation_accuracy": best_accuracy,

            "best_validation_precision": best_precision,

            "best_validation_recall": best_recall,

            "best_validation_f1": best_val_f1

        })

        # =====================================
        # Save Best Model
        # =====================================

        mlflow.sklearn.log_model(

            best_model,

            artifact_path="best_model"

        )

        return (

            best_model,

            best_params,

            best_f1,

            study

        )

In [ ]:
best_model, best_params, best_f1, study = tune_logistic(
    n_trials=100
)

<h2> 2nd model SVC

In [ ]:
from sklearn.svm import LinearSVC

def objective_svc(trial):

    with mlflow.start_run(
        run_name=f"Trial_{trial.number}",
        nested=True
    ):

        start_time = time.time()

        mlflow.set_tag(
            "model",
            "LinearSVC"
        )

        # Hyperparameters

        C = trial.suggest_float(
            "C",
            1e-3,
            100,
            log=True
        )

        loss = trial.suggest_categorical(

            "loss",

            [

                "hinge",

                "squared_hinge"

            ]

        )

        class_weight = trial.suggest_categorical(

            "class_weight",

            [

                None,

                "balanced"

            ]

        )

        # Model

        model = LinearSVC(

            C=C,

            loss=loss,

            class_weight=class_weight,

            max_iter=5000,

            random_state=42

        )

        # Train

        model.fit(

            X_train,

            y_train

        )

        # Predict

        y_pred = model.predict(

            X_val

        )

        # Metrics

        accuracy = accuracy_score(

            y_val,

            y_pred

        )

        precision = precision_score(

            y_val,

            y_pred

        )

        recall = recall_score(

            y_val,

            y_pred

        )

        f1 = f1_score(

            y_val,

            y_pred

        )

        training_time = time.time() - start_time

        # MLflow logging

        mlflow.log_params({

            "C": C,

            "loss": loss,

            "class_weight": class_weight,

            "max_iter": 5000

        })

        mlflow.log_metrics({

            "accuracy": accuracy,

            "precision": precision,

            "recall": recall,

            "f1_score": f1,

            "training_time_seconds": training_time

        })

        return f1

In [ ]:
def tune_svc(n_trials=50):

    with mlflow.start_run(

        run_name="LinearSVC_Optuna"

    ):

        mlflow.log_param(

            "embedding_model",

            "all-MiniLM-L6-v2"

        )

        mlflow.log_param(

            "embedding_dimension",

            384

        )

        mlflow.log_param(

            "n_trials",

            n_trials

        )

        study = optuna.create_study(

            direction="maximize",

            study_name="LinearSVC"

        )


        study.optimize(

            objective_svc,

            n_trials=n_trials,

            catch=(Exception,)

            )

        best_params = study.best_params

        best_f1 = study.best_value

        print("\nBest Trial")

        print("----------------------")

        print(

            "Best Validation F1:",

            best_f1

        )

        print("\nBest Parameters:")

        for k, v in best_params.items():

            print(f"{k} : {v}")

        # Train Best Model

        best_model = LinearSVC(

            **best_params,

            max_iter=5000,

            random_state=42

        )

        best_model.fit(

            X_train,

            y_train

        )

        # Validation Metrics

        y_pred_best = best_model.predict(

            X_val

        )

        best_accuracy = accuracy_score(

            y_val,

            y_pred_best

        )

        best_precision = precision_score(

            y_val,

            y_pred_best

        )

        best_recall = recall_score(

            y_val,

            y_pred_best

        )

        best_val_f1 = f1_score(

            y_val,

            y_pred_best

        )

        mlflow.log_params({

            f"best_{k}": v

            for k, v in best_params.items()

        })

        mlflow.log_metrics({

            "best_validation_accuracy": best_accuracy,

            "best_validation_precision": best_precision,

            "best_validation_recall": best_recall,

            "best_validation_f1": best_val_f1

        })

        mlflow.sklearn.log_model(

            best_model,

            artifact_path="best_model"

        )

        return (

            best_model,

            best_params,

            best_f1,

            study

        )

In [ ]:
best_model, best_params, best_f1, study = tune_svc(
    n_trials=50
)

<h1>XG boost

In [ ]:
from xgboost import XGBClassifier

def objective_xgb(trial):

    with mlflow.start_run(
        run_name=f"Trial_{trial.number}",
        nested=True
    ):

        start_time = time.time()

        mlflow.set_tag(
            "model",
            "XGBoost"
        )

        model = XGBClassifier(

            n_estimators=trial.suggest_int(
                "n_estimators",
                100,
                600
            ),

            max_depth=trial.suggest_int(
                "max_depth",
                3,
                10
            ),

            learning_rate=trial.suggest_float(
                "learning_rate",
                0.01,
                0.3,
                log=True
            ),

            subsample=trial.suggest_float(
                "subsample",
                0.5,
                1.0
            ),

            colsample_bytree=trial.suggest_float(
                "colsample_bytree",
                0.5,
                1.0
            ),

            reg_alpha=trial.suggest_float(
                "reg_alpha",
                0,
                10
            ),

            reg_lambda=trial.suggest_float(
                "reg_lambda",
                0,
                10
            ),

            objective='binary:logistic',

            eval_metric='logloss',

            random_state=42

        )

        model.fit(

            X_train,

            y_train

        )

        y_pred = model.predict(

            X_val

        )

        accuracy = accuracy_score(
            y_val,
            y_pred
        )

        precision = precision_score(
            y_val,
            y_pred
        )

        recall = recall_score(
            y_val,
            y_pred
        )

        f1 = f1_score(
            y_val,
            y_pred
        )

        training_time = time.time() - start_time

        mlflow.log_params(

            model.get_params()

        )

        mlflow.log_metrics({

            "accuracy": accuracy,

            "precision": precision,

            "recall": recall,

            "f1_score": f1,

            "training_time_seconds": training_time

        })

        return f1

In [ ]:
def tune_xgb(n_trials=75):

    with mlflow.start_run(

        run_name="XGBoost_Optuna"

    ):

        mlflow.log_param(

            "embedding_model",

            "all-MiniLM-L6-v2"

        )

        mlflow.log_param(

            "embedding_dimension",

            384

        )

        mlflow.log_param(

            "n_trials",

            n_trials

        )

        study = optuna.create_study(

            direction="maximize",

            study_name="XGBoost"

        )

        study.optimize(

            objective_xgb,

            n_trials=n_trials,

            catch=(Exception,)

        )

        best_params = study.best_params

        best_f1 = study.best_value

        print("\nBest Trial")

        print("----------------------")

        print(

            "Best Validation F1:",

            best_f1

        )

        print("\nBest Parameters:")

        for k, v in best_params.items():

            print(f"{k} : {v}")

        best_model = XGBClassifier(

            **best_params,

            objective='binary:logistic',

            eval_metric='logloss',

            random_state=42

        )

        best_model.fit(

            X_train,

            y_train

        )

        y_pred_best = best_model.predict(

            X_val

        )

        best_accuracy = accuracy_score(

            y_val,

            y_pred_best

        )

        best_precision = precision_score(

            y_val,

            y_pred_best

        )

        best_recall = recall_score(

            y_val,

            y_pred_best

        )

        best_val_f1 = f1_score(

            y_val,

            y_pred_best

        )

        mlflow.log_metrics({

            "best_validation_accuracy": best_accuracy,

            "best_validation_precision": best_precision,

            "best_validation_recall": best_recall,

            "best_validation_f1": best_val_f1

        })

        mlflow.sklearn.log_model(

            best_model,

            artifact_path="best_model"

        )

        return (

            best_model,

            best_params,

            best_f1,

            study

        )

In [ ]:
best_model, best_params, best_f1, study = tune_xgb(
    n_trials=75
)

<h1>light gbm

In [ ]:
from lightgbm import LGBMClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

import optuna
import mlflow
import mlflow.sklearn
import time

In [ ]:
def objective_lgbm(trial):

    with mlflow.start_run(
        run_name=f"Trial_{trial.number}",
        nested=True
    ):

        start_time = time.time()

        mlflow.set_tag(
            "model",
            "LightGBM"
        )

        model = LGBMClassifier(

            n_estimators=trial.suggest_int(
                "n_estimators",
                100,
                600
            ),

            learning_rate=trial.suggest_float(
                "learning_rate",
                0.01,
                0.3,
                log=True
            ),

            max_depth=trial.suggest_int(
                "max_depth",
                3,
                15
            ),

            num_leaves=trial.suggest_int(
                "num_leaves",
                20,
                150
            ),

            min_child_samples=trial.suggest_int(
                "min_child_samples",
                5,
                50
            ),

            subsample=trial.suggest_float(
                "subsample",
                0.5,
                1.0
            ),

            colsample_bytree=trial.suggest_float(
                "colsample_bytree",
                0.5,
                1.0
            ),

            reg_alpha=trial.suggest_float(
                "reg_alpha",
                0,
                10
            ),

            reg_lambda=trial.suggest_float(
                "reg_lambda",
                0,
                10
            ),

            random_state=42,

            verbose=-1
        )

        model.fit(
            X_train,
            y_train
        )

        y_pred = model.predict(
            X_val
        )

        accuracy = accuracy_score(
            y_val,
            y_pred
        )

        precision = precision_score(
            y_val,
            y_pred
        )

        recall = recall_score(
            y_val,
            y_pred
        )

        f1 = f1_score(
            y_val,
            y_pred
        )

        training_time = time.time() - start_time

        mlflow.log_params(
            model.get_params()
        )

        mlflow.log_metrics({

            "accuracy": accuracy,

            "precision": precision,

            "recall": recall,

            "f1_score": f1,

            "training_time_seconds": training_time

        })

        return f1

In [ ]:
def tune_lgbm(n_trials=75):

    with mlflow.start_run(

        run_name="LightGBM_Optuna"

    ):

        mlflow.log_param(

            "embedding_model",

            "all-MiniLM-L6-v2"

        )

        mlflow.log_param(

            "embedding_dimension",

            384

        )

        mlflow.log_param(

            "n_trials",

            n_trials

        )

        study = optuna.create_study(

            direction="maximize",

            study_name="LightGBM"

        )

        study.optimize(

            objective_lgbm,

            n_trials=n_trials,

            catch=(Exception,)

        )

        best_params = study.best_params

        best_f1 = study.best_value

        print("\nBest Trial")

        print("----------------------")

        print(

            "Best Validation F1:",

            best_f1

        )

        print("\nBest Parameters:")

        for k, v in best_params.items():

            print(f"{k} : {v}")

        best_model = LGBMClassifier(

            **best_params,

            random_state=42,

            verbose=-1

        )

        best_model.fit(

            X_train,

            y_train

        )

        y_pred_best = best_model.predict(

            X_val

        )

        best_accuracy = accuracy_score(

            y_val,

            y_pred_best

        )

        best_precision = precision_score(

            y_val,

            y_pred_best

        )

        best_recall = recall_score(

            y_val,

            y_pred_best

        )

        best_val_f1 = f1_score(

            y_val,

            y_pred_best

        )

        mlflow.log_metrics({

            "best_validation_accuracy": best_accuracy,

            "best_validation_precision": best_precision,

            "best_validation_recall": best_recall,

            "best_validation_f1": best_val_f1

        })

        mlflow.sklearn.log_model(

            best_model,

            artifact_path="best_model"

        )

        return (

            best_model,

            best_params,

            best_f1,

            study

        )

In [ ]:
best_model, best_params, best_f1, study = tune_lgbm(
    n_trials=75
)

<h1>MLP classifier

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

import mlflow
import time


def objective_mlp(trial):

    with mlflow.start_run(
        run_name=f"Trial_{trial.number}",
        nested=True
    ):

        start_time = time.time()

        mlflow.set_tag(
            "model",
            "MLPClassifier"
        )

        hidden_layer_sizes = trial.suggest_categorical(

            "hidden_layer_sizes",

            [

                (128,),

                (256,),

                (128, 64),

                (256, 128)

            ]

        )

        activation = trial.suggest_categorical(

            "activation",

            [

                "relu",

                "tanh"

            ]

        )

        alpha = trial.suggest_float(

            "alpha",

            1e-5,

            1e-2,

            log=True

        )

        learning_rate_init = trial.suggest_float(

            "learning_rate_init",

            1e-4,

            1e-2,

            log=True

        )

        batch_size = trial.suggest_categorical(

            "batch_size",

            [

                32,

                64,

                128

            ]

        )

        model = MLPClassifier(

            hidden_layer_sizes=hidden_layer_sizes,

            activation=activation,

            alpha=alpha,

            learning_rate_init=learning_rate_init,

            batch_size=batch_size,

            max_iter=100,

            early_stopping=True,

            n_iter_no_change=10,

            random_state=42

        )

        model.fit(

            X_train,

            y_train

        )

        y_pred = model.predict(

            X_val

        )

        accuracy = accuracy_score(

            y_val,

            y_pred

        )

        precision = precision_score(

            y_val,

            y_pred

        )

        recall = recall_score(

            y_val,

            y_pred

        )

        f1 = f1_score(

            y_val,

            y_pred

        )

        training_time = time.time() - start_time

        mlflow.log_params({

            "hidden_layer_sizes": hidden_layer_sizes,

            "activation": activation,

            "alpha": alpha,

            "learning_rate_init": learning_rate_init,

            "batch_size": batch_size,

            "max_iter": 100

        })

        mlflow.log_metrics({

            "accuracy": accuracy,

            "precision": precision,

            "recall": recall,

            "f1_score": f1,

            "training_time_seconds": training_time

        })

        return f1

In [ ]:
def tune_mlp(n_trials=40):

    with mlflow.start_run(

        run_name="MLPClassifier_Optuna"

    ):

        mlflow.log_param(

            "embedding_model",

            "all-MiniLM-L6-v2"

        )

        mlflow.log_param(

            "embedding_dimension",

            384

        )

        mlflow.log_param(

            "n_trials",

            n_trials

        )

        study = optuna.create_study(

            direction="maximize",

            study_name="MLPClassifier"

        )

        study.optimize(

            objective_mlp,

            n_trials=n_trials

        )

        print("\nBest Trial")

        print("----------------")

        print(

            "Best Validation F1:",

            study.best_value

        )

        print("\nBest Parameters:")

        for k, v in study.best_params.items():

            print(f"{k} : {v}")

        return study

In [ ]:
study_mlp = tune_mlp(
    n_trials=40
)